# Re-train Exp C (Fixed) + Multi-Seed Exp B/A/D + Per-Class AP Chart

## Thứ tự chạy (BẮT BUỘC):
1. **Cell 1** — Install & GPU check
2. **Cell 2** — Clone repo → lấy `src/`, `configs/` đã fix
3. **Cell 3** — Mount COCO 2017 & tạo subset
4. **Cell 4** — Verify tất cả fixes (ASL, evaluate, configs)
5. **Cell 5** — Re-train Exp C (ưu tiên nhất → expect mAP ~0.75)
6. **Cell 6** — Multi-seed Exp B (3 seeds → mean ± std)
7. **Cell 7** — Multi-seed Exp A & D
8. **Cell 8** — Multi-seed Exp C (nếu muốn báo cáo confidence interval)
9. **Cell 9** — Per-class AP bar chart
10. **Cell 10** — Ablation summary table

## ⚠️ ROOT CAUSE của mAP thấp (0.56 vs 0.75):
```
evaluate.py (master bug):
    with torch.amp.autocast('cuda', ...):   ← autocast → fp16 sigmoid
        logits = model(...)
→ fp16 precision thấp → sigmoid scores bị quantize → mAP tính sai

evaluate.py (fixed):
    logits = model(imgs.to(device))         ← fp32, không autocast
→ sigmoid chính xác → mAP đúng ~0.75
```


## Cell 1: Install & GPU Check

In [ ]:
!pip install timm pycocotools torchmetrics scikit-learn matplotlib seaborn pyyaml tqdm -q
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')


## Cell 2: Clone Repository từ GitHub

In [ ]:
import os, sys
from pathlib import Path

REPO_URL = 'https://github.com/Thinh59/ECAAL.git'  # ← đổi nếu URL khác
REPO_DIR = Path('/kaggle/working/ECAAL')

if REPO_DIR.exists():
    print('Repo đã tồn tại — pulling latest...')
    os.system(f'git -C {REPO_DIR} pull')
else:
    print(f'Cloning {REPO_URL} ...')
    ret = os.system(f'git clone {REPO_URL} {REPO_DIR}')
    if ret != 0:
        raise RuntimeError('git clone thất bại! Kiểm tra REPO_URL.')

required = [
    'src/train.py', 'src/losses.py', 'src/models.py',
    'src/dataset.py', 'src/evaluate.py', 'src/cbam.py', 'src/utils.py',
    'configs/exp_A_resnet_bce.yaml', 'configs/exp_B_resnet_asl.yaml',
    'configs/exp_C_efficientnet_cbam_asl.yaml', 'configs/exp_D_resnet_focal.yaml',
    'configs/exp_E_resnet_cbam_asl.yaml', 'configs/exp_F_efficientnet_asl.yaml',
]
all_ok = True
for rel in required:
    p = REPO_DIR / rel
    ok = p.exists()
    print(f"  {'OK' if ok else 'MISSING'} {rel}")
    if not ok:
        all_ok = False

if not all_ok:
    raise FileNotFoundError('Một số file thiếu!')
print('\nRepository OK!')


## Cell 3: Mount COCO 2017 & Tạo Subset

**Trước khi chạy:** Notebook → **+ Add Data** → search `coco-2017-dataset` (awsaf49) → Add.
Dataset sẽ mount tại `/kaggle/input/coco-2017-dataset/`.


In [ ]:
import json
from pathlib import Path

COCO_ROOT  = Path('/kaggle/input/coco-2017-dataset')
SUBSET_DIR = Path('/kaggle/working/data/coco_subset')
SUBSET_DIR.mkdir(parents=True, exist_ok=True)
os.makedirs('/kaggle/working/outputs', exist_ok=True)

# Verify COCO mount
for name, p in [
    ('train annotations', COCO_ROOT / 'annotations' / 'instances_train2017.json'),
    ('val annotations',   COCO_ROOT / 'annotations' / 'instances_val2017.json'),
    ('train2017/',        COCO_ROOT / 'train2017'),
    ('val2017/',          COCO_ROOT / 'val2017'),
]:
    print(f"  {'OK' if p.exists() else 'MISSING'} {name}")

if not (COCO_ROOT / 'annotations' / 'instances_train2017.json').exists():
    raise FileNotFoundError('COCO chưa mount! → Notebook → + Add Data → coco-2017-dataset')

# Tạo subset (skip nếu đã có)
train_f = SUBSET_DIR / 'subset_train_ids.json'
val_f   = SUBSET_DIR / 'subset_val_ids.json'
if train_f.exists() and val_f.exists():
    print(f"\n✅ Subset đã có — skip (train={len(json.load(open(train_f)))}, val={len(json.load(open(val_f)))})")
else:
    sys.path.insert(0, str(REPO_DIR / 'src'))
    from dataset import create_coco_subset
    create_coco_subset(str(COCO_ROOT), str(SUBSET_DIR), num_train=16000, num_val=4000, seed=42)
    print("✅ Subset created!")


## Cell 4: Verify All Fixes

In [ ]:
import sys, inspect, yaml
sys.path.insert(0, str(REPO_DIR / 'src'))

from losses   import AsymmetricLoss
from evaluate import evaluate_model

errors = []

# Fix 1: ASL correct shifting
src_asl = inspect.getsource(AsymmetricLoss.forward)
ok1 = 'xs_pos_shifted = (xs_pos - self.clip).clamp(min=0)' in src_asl
ok2 = 'xs_neg_shifted = (xs_neg + self.clip)' not in src_asl
print(f"  {'✅' if ok1 else '❌'} ASL: correct probability shifting")
print(f"  {'✅' if ok2 else '❌'} ASL: old bug absent")
if not ok1 or not ok2: errors.append('ASL bug')

# Fix 2: evaluate.py không có autocast
src_eval = inspect.getsource(evaluate_model)
ok3 = 'autocast' not in src_eval
print(f"  {'✅' if ok3 else '❌'} evaluate_model: NO autocast (fp32 → accurate mAP)")
if not ok3: errors.append('evaluate autocast bug')

# Fix 3: exp_C không có eps=1e-5 thừa
cfg_c = yaml.safe_load(open(REPO_DIR / 'configs/exp_C_efficientnet_cbam_asl.yaml'))
ok4 = 'eps' not in cfg_c.get('loss', {})
print(f"  {'✅' if ok4 else '⚠️ '} exp_C config: {'no eps (correct)' if ok4 else 'has eps=' + str(cfg_c['loss'].get('eps'))}")

# Fix 4: exp_E & exp_F configs tồn tại và đúng backbone
cfg_e = yaml.safe_load(open(REPO_DIR / 'configs/exp_E_resnet_cbam_asl.yaml'))
cfg_f = yaml.safe_load(open(REPO_DIR / 'configs/exp_F_efficientnet_asl.yaml'))
ok5 = cfg_e['model']['backbone'] == 'resnet50'    and cfg_e['model']['use_cbam'] == True
ok6 = cfg_f['model']['backbone'] == 'efficientnet_b0' and cfg_f['model']['use_cbam'] == False
print(f"  {'✅' if ok5 else '❌'} exp_E: resnet50 + CBAM=True")
print(f"  {'✅' if ok6 else '❌'} exp_F: efficientnet_b0 + CBAM=False")
if not ok5: errors.append('exp_E config')
if not ok6: errors.append('exp_F config')

# ASL NaN test
import torch
asl  = AsymmetricLoss(gamma_pos=0, gamma_neg=4, clip=0.05)
lg   = torch.tensor([[10., -10., 5., -5.]] * 8)
tg   = torch.randint(0, 2, (8, 4)).float()
loss = asl(lg, tg)
ok7  = not torch.isnan(loss)
print(f"  {'✅' if ok7 else '❌'} ASL: no NaN on saturated logits (loss={loss.item():.4f})")
if not ok7: errors.append('ASL NaN')

if errors:
    raise RuntimeError(f'❌ Fixes chưa được apply: {errors}')
print('\n🎉 ALL FIXES VERIFIED — Ready to train!')


## Cell 5: Re-train Exp C — EfficientNet-B0 + CBAM + ASL ⭐

**Expected mAP: ~0.75** (vs 0.56 với master bug evaluate.py + eps=1e-5)


In [ ]:
import time
from train import run
import yaml

CONFIGS    = REPO_DIR / 'configs'
OUTPUT_DIR = Path('/kaggle/working/outputs')

cfg_c_path = CONFIGS / 'exp_C_efficientnet_cbam_asl.yaml'
cfg_c = yaml.safe_load(open(cfg_c_path))
print(f'Config: {cfg_c_path}')
print(f"  backbone: {cfg_c['model']['backbone']}, cbam={cfg_c['model']['use_cbam']}")
print(f"  lr={cfg_c['optimizer']['lr']}, loss={cfg_c['loss']['name']}")
print(f"  eps in loss: {cfg_c['loss'].get('eps', 'default 1e-8')}")

t0 = time.time()
best_c = run(str(cfg_c_path))
elapsed = (time.time() - t0) / 3600

print(f'\n[Done] Exp C | mAP={best_c:.4f} | {elapsed:.1f}h')
print(f'  Old (master bug): 0.5626')
print(f'  Fixed (expected): ~0.7500')
print(f'  Actual result:     {best_c:.4f}')
if best_c >= 0.73:
    print('  ✅ Fix confirmed!')
else:
    print('  ⚠️  Thấp hơn expected — kiểm tra lại fix')


## Cell 6: Multi-Seed Exp B — ResNet50 + ASL (seeds 42, 123, 2025)

Chạy 3 seeds để báo cáo `mean ± std` cho paper.


In [ ]:
import yaml, copy, numpy as np, json
from train import run

SEEDS = [42, 123, 2025]
cfg_b_path = CONFIGS / 'exp_B_resnet_asl.yaml'
cfg_b_base = yaml.safe_load(open(cfg_b_path))

maps_b = {}
for seed in SEEDS:
    cfg_tmp = copy.deepcopy(cfg_b_base)
    cfg_tmp['seed'] = seed
    cfg_tmp['output_dir'] = f'/kaggle/working/outputs_multiseed/expB_seed{seed}'
    tmp = f'/kaggle/working/tmp_B_s{seed}.yaml'
    with open(tmp, 'w') as f:
        yaml.dump(cfg_tmp, f)
    print(f'\n--- Exp B | Seed {seed} ---')
    maps_b[seed] = run(tmp)
    print(f'  mAP = {maps_b[seed]:.4f}')

vals = list(maps_b.values())
print(f'\nExp B Multi-Seed: mean={np.mean(vals):.4f} ± std={np.std(vals):.4f}')
print(f'  Per seed: {maps_b}')


## Cell 7: Multi-Seed Exp A (BCE) & Exp D (Focal)

In [ ]:
SEEDS = [42, 123, 2025]
multiseed = {}

for exp_tag, cfg_file in [
    ('A', CONFIGS / 'exp_A_resnet_bce.yaml'),
    ('D', CONFIGS / 'exp_D_resnet_focal.yaml'),
]:
    base = yaml.safe_load(open(cfg_file))
    seed_maps = {}
    for seed in SEEDS:
        cfg_tmp = copy.deepcopy(base)
        cfg_tmp['seed'] = seed
        cfg_tmp['output_dir'] = f'/kaggle/working/outputs_multiseed/exp{exp_tag}_seed{seed}'
        tmp = f'/kaggle/working/tmp_{exp_tag}_s{seed}.yaml'
        with open(tmp, 'w') as f:
            yaml.dump(cfg_tmp, f)
        print(f'Exp {exp_tag} | Seed {seed}')
        seed_maps[seed] = run(tmp)
        print(f'  mAP = {seed_maps[seed]:.4f}')
    vals = list(seed_maps.values())
    multiseed[f'Exp_{exp_tag}'] = {'mean': np.mean(vals), 'std': np.std(vals), 'per_seed': seed_maps}
    print(f'Exp {exp_tag}: mean={np.mean(vals):.4f} ± {np.std(vals):.4f}')

# Thêm Exp B vào multiseed
multiseed['Exp_B'] = {'mean': np.mean(list(maps_b.values())), 'std': np.std(list(maps_b.values())), 'per_seed': maps_b}

with open('/kaggle/working/outputs/multiseed_results.json', 'w') as f:
    json.dump(multiseed, f, indent=2)
print('\n✅ Multi-seed results saved to outputs/multiseed_results.json')


## Cell 8: Multi-Seed Exp C (Optional — confidence interval)

In [ ]:
SEEDS = [42, 123, 2025]
cfg_c_base = yaml.safe_load(open(CONFIGS / 'exp_C_efficientnet_cbam_asl.yaml'))

maps_c = {}
for seed in SEEDS:
    cfg_tmp = copy.deepcopy(cfg_c_base)
    cfg_tmp['seed'] = seed
    cfg_tmp['output_dir'] = f'/kaggle/working/outputs_multiseed/expC_seed{seed}'
    tmp = f'/kaggle/working/tmp_C_s{seed}.yaml'
    with open(tmp, 'w') as f:
        yaml.dump(cfg_tmp, f)
    print(f'\n--- Exp C | Seed {seed} ---')
    maps_c[seed] = run(tmp)
    print(f'  mAP = {maps_c[seed]:.4f}')

vals = list(maps_c.values())
print(f'\nExp C Multi-Seed: mean={np.mean(vals):.4f} ± std={np.std(vals):.4f}')


## Cell 9: Per-Class AP Bar Chart (Figure 2 — Paper)

So sánh AP từng class giữa Exp A (BCE), Exp B (ASL), Exp C (EffNet+CBAM+ASL).


In [ ]:
import json, numpy as np, matplotlib.pyplot as plt

BASE = '/kaggle/working/outputs'

def load_best(path):
    recs = json.load(open(path))
    return max(recs, key=lambda r: r.get('mAP', 0))

COCO_CLASSES = [
    'person','bicycle','car','motorcycle','airplane','bus','train','truck',
    'boat','traffic light','fire hydrant','stop sign','parking meter','bench',
    'bird','cat','dog','horse','sheep','cow','elephant','bear','zebra','giraffe',
    'backpack','umbrella','handbag','tie','suitcase','frisbee','skis','snowboard',
    'sports ball','kite','baseball bat','baseball glove','skateboard','surfboard',
    'tennis racket','bottle','wine glass','cup','fork','knife','spoon','bowl',
    'banana','apple','sandwich','orange','broccoli','carrot','hot dog','pizza',
    'donut','cake','chair','couch','potted plant','bed','dining table','toilet',
    'tv','laptop','mouse','remote','keyboard','cell phone','microwave','oven',
    'toaster','sink','refrigerator','book','clock','vase','scissors','teddy bear',
    'hair drier','toothbrush',
]

loaded = {}
for tag, folder in [('A: BCE', 'exp_A_resnet_bce'), ('B: ASL', 'exp_B_resnet_asl'),
                    ('C: EffNet+CBAM+ASL', 'exp_C_efficientnet_cbam_asl')]:
    p = f'{BASE}/{folder}/log.json'
    if os.path.exists(p):
        loaded[tag] = np.array(load_best(p)['AP_per_class'])

if len(loaded) < 2:
    print("⚠️  Cần ít nhất 2 experiments để vẽ biểu đồ")
else:
    n = min(len(v) for v in loaded.values())
    x = np.arange(n)
    width = 0.25
    colors = ['#e74c3c', '#3498db', '#2ecc71']

    fig, ax = plt.subplots(figsize=(22, 6))
    for i, (tag, aps) in enumerate(loaded.items()):
        ax.bar(x + i*width, aps[:n], width, label=tag, color=colors[i], alpha=0.85)

    ax.set_xticks(x + width)
    ax.set_xticklabels(COCO_CLASSES[:n], rotation=90, fontsize=7)
    ax.set_ylabel('AP', fontsize=10)
    ax.set_title('Per-Class AP Comparison', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    plt.tight_layout()
    plt.savefig(f'{BASE}/per_class_ap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Chart saved to {BASE}/per_class_ap.png")


## Cell 10: Ablation Summary Table (Paper)

In [ ]:
import json, os, pandas as pd

BASE = '/kaggle/working/outputs'
exps = [
    ('A', 'ResNet50',       'BCE',           False, 'exp_A_resnet_bce'),
    ('D', 'ResNet50',       'Focal',         False, 'exp_D_resnet_focal'),
    ('B', 'ResNet50',       'ASL',           False, 'exp_B_resnet_asl'),
    ('E', 'ResNet50',       'ASL',           True,  'exp_E_resnet_cbam_asl'),
    ('F', 'EfficientNet-B0','ASL',           False, 'exp_F_efficientnet_asl'),
    ('C', 'EfficientNet-B0','ASL',           True,  'exp_C_efficientnet_cbam_asl'),
]

rows = []
base_map = None
for exp_id, backbone, loss, cbam, folder in exps:
    p = f'{BASE}/{folder}/log.json'
    if not os.path.exists(p):
        print(f'Missing: {p}')
        continue
    recs = json.load(open(p))
    best = max(recs, key=lambda r: r.get('mAP', 0))
    if exp_id == 'A':
        base_map = best['mAP']
    delta = f"+{best['mAP']-base_map:.4f}" if base_map and exp_id != 'A' else '—'
    rows.append({
        'Exp': exp_id,
        'Backbone': backbone,
        'Loss': loss,
        'CBAM': 'Y' if cbam else 'N',
        'mAP': f"{best['mAP']:.4f}",
        'Macro F1': f"{best['macro_f1']:.4f}",
        'Δ mAP': delta,
        'Best Epoch': best['epoch'],
    })

print("\n" + "="*85)
print("ABLATION STUDY — Full Results")
print("="*85)
print(pd.DataFrame(rows).to_string(index=False))
print("="*85)
